# BookWise — Content-Based Filtering
Dataset: GoodBooks-10k
Metode: TF-IDF + Cosine Similarity

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib, os

DATA_DIR  = '../../data'
MODEL_DIR = '../model'
os.makedirs(MODEL_DIR, exist_ok=True)

# GoodBooks-10k
books   = pd.read_csv(f'{DATA_DIR}/books.csv')
ratings = pd.read_csv(f'{DATA_DIR}/ratings.csv')

books = books[['book_id','isbn','title','authors','original_publication_year','image_url','average_rating']].copy()
books.dropna(subset=['book_id','title'], inplace=True)
books.drop_duplicates(subset='book_id', inplace=True)
books['isbn'] = books['isbn'].fillna(books['book_id'].astype(str))
books['book_id'] = books['book_id'].astype(int)

print(f'Books loaded: {len(books):,}')

In [ ]:
# Buat fitur teks dari title + authors
books['features'] = books['title'].fillna('') + ' ' + books['authors'].fillna('')

tfidf     = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_mat = tfidf.fit_transform(books['features'])
print(f'TF-IDF matrix shape: {tfidf_mat.shape}')

cosine_sim    = cosine_similarity(tfidf_mat, tfidf_mat)
book_id_index = {int(bid): idx for idx, bid in enumerate(books['book_id'])}
isbn_index    = {str(isbn): idx for idx, isbn in enumerate(books['isbn'])}
print(f'Cosine sim matrix: {cosine_sim.shape}')

In [ ]:
def get_recommendations(book_id, top_k=10):
    idx = book_id_index.get(int(book_id))
    if idx is None:
        return pd.DataFrame()
    sim_scores = sorted(enumerate(cosine_sim[idx]), key=lambda x: x[1], reverse=True)[1:top_k+1]
    return books.iloc[[i[0] for i in sim_scores]][['book_id','title','authors','average_rating']]

# Test dengan buku pertama
sample = books.iloc[0]
print(f'Rekomendasi untuk: "{sample["title"]}" oleh {sample["authors"]}')
print(get_recommendations(sample['book_id'], 5).to_string(index=False))

In [ ]:
# Evaluasi Precision@10 & Recall@10
high_ratings = ratings[ratings['rating'] >= 4]
sample_users = high_ratings['user_id'].value_counts().head(300).index

p_scores, r_scores = [], []
for user in sample_users:
    user_books = high_ratings[high_ratings['user_id'] == user]['book_id'].tolist()
    if len(user_books) < 2:
        continue
    recs = get_recommendations(user_books[0], 10)
    if recs.empty:
        continue
    rec_ids  = set(recs['book_id'].tolist())
    relevant = set(user_books[1:])
    hit = len(rec_ids & relevant)
    p_scores.append(hit / 10)
    r_scores.append(hit / len(relevant) if relevant else 0)

print(f'Precision@10: {np.mean(p_scores):.4f}')
print(f'Recall@10:    {np.mean(r_scores):.4f}')
print(f'Evaluated on {len(p_scores)} users')

In [ ]:
# Visualisasi distribusi cosine similarity
sample_sims = cosine_sim[0][1:]
plt.figure(figsize=(8, 4))
plt.hist(sample_sims, bins=50, color='steelblue', edgecolor='white')
plt.xlabel('Cosine Similarity Score')
plt.ylabel('Frekuensi')
plt.title(f'Distribusi Similarity untuk "{books.iloc[0]["title"]}"')
plt.tight_layout()
plt.savefig('cosine_sim_distribution.png', dpi=150)
plt.show()

# Simpan model
cb_model = {'cosine_sim': cosine_sim, 'book_id_index': book_id_index, 'isbn_index': isbn_index}
joblib.dump(cb_model, f'{MODEL_DIR}/content_based.pkl')
books.to_pickle(f'{MODEL_DIR}/books.pkl')
print('✅ Model saved.')